# Figure 5 -- De novo generation

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.lines import Line2D
from matplotlib.colors import LinearSegmentedColormap, TwoSlopeNorm
from matplotlib.ticker import MaxNLocator
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-paper')
plt.rcParams.update({
    'font.size':7,'axes.titlesize':7,'axes.labelsize':7,
    'xtick.labelsize':6,'ytick.labelsize':6,'legend.fontsize':6,
    'axes.linewidth':0.5,'axes.edgecolor':'#888888',
    'xtick.color':'#555555','ytick.color':'#555555',
    'axes.labelcolor':'#333333','font.family':'sans-serif',
    'font.sans-serif':['Arial','DejaVu Sans'],
    'figure.dpi':300,'savefig.dpi':600,
    'xtick.major.width':0.5,'ytick.major.width':0.5,
    'xtick.major.size':2.5,'ytick.major.size':2.5,
})

DATA='/home/pszymczak/code/omegamp-dashboard/data/'
BENCH='/home/pszymczak/code/omegamp-dashboard/benchmark/'
ref=pd.read_csv(DATA+'omegamp_reference_table.csv')
mic=pd.read_csv(DATA+'mic.csv'); hc50=pd.read_csv(DATA+'hc50.csv')
cc50=pd.read_csv(DATA+'cc50.csv'); npn=pd.read_csv(DATA+'npn.csv')
disc=pd.read_csv(DATA+'disc.csv'); bestsel=pd.read_csv(DATA+'bestsel.csv')

HYDRO_SCALE={'A':0.62,'R':-2.53,'N':-0.78,'D':-0.90,'C':0.29,'Q':-0.85,'E':-0.74,
             'G':0.48,'H':-0.40,'I':1.38,'L':1.06,'K':-1.50,'M':0.64,'F':1.19,
             'P':0.12,'S':-0.18,'T':-0.05,'W':0.81,'Y':0.26,'V':1.08}
CHARGE_MAP={'K':1,'R':1,'H':0.5,'D':-1,'E':-1}
try:
    rr=pd.read_csv(DATA+'reference_amps.csv'); ref_amps_seq=rr[rr['IsAMP']==1]['Sequence'].tolist()
except FileNotFoundError:
    ref_amps_seq=ref['sequence'].dropna().tolist()
ref_charges=[sum(CHARGE_MAP.get(aa,0) for aa in s.upper()) for s in ref_amps_seq]
ref_hydros=[np.mean([HYDRO_SCALE.get(aa,0) for aa in s.upper()]) for s in ref_amps_seq if len(s)>0]
ref_amp_props=pd.DataFrame({'charge':ref_charges[:len(ref_hydros)],'hydrophobicity':ref_hydros})

df=ref.merge(hc50,on='short_name',how='left').merge(cc50,on='short_name',how='left')
df=df.merge(npn.rename(columns={'MaxRel':'NPN','AUC':'NPN_AUC'}),on='short_name',how='left')
df=df.merge(disc.rename(columns={'MaxRel':'DiSC','AUC':'DiSC_AUC'}),on='short_name',how='left')
mic_strains=mic.columns[1:]; mic_vals=mic[mic_strains].astype(float)
mic['n_strains_le2']=(mic_vals<=2).sum(axis=1); mic['n_strains_le4']=(mic_vals<=4).sum(axis=1)
mic['mic50']=mic_vals.fillna(128).clip(upper=128).median(axis=1)
df=df.merge(mic[['short_name','n_strains_le2','n_strains_le4','mic50']],on='short_name',how='left')
df['HC50']=df['HC50'].clip(0.1,128); df['CC50']=df['CC50'].clip(0.1,128)

de_novo=df[df['category']=='de_novo'].copy()
dp=de_novo[de_novo['generation_mode']=='OmegAMP-P']
dt=de_novo[de_novo['generation_mode']=='OmegAMP-T']
mic_dn=mic[mic['short_name'].isin(de_novo['short_name'])]

mic_cmap=LinearSegmentedColormap.from_list('mic_rb',
    [(0.0,'#b2182b'),(0.25,'#ef8a62'),(0.5,'#f7f7f7'),(0.75,'#67a9cf'),(1.0,'#2166ac')])
mic_norm=TwoSlopeNorm(vmin=0,vcenter=32,vmax=64)

# De novo leads: full names (no prototype to abbreviate)
LEADS={
    '\u03a9-DP-19':'\u03a9-DP-19',
    '\u03a9-DT-10':'\u03a9-DT-10',
    '\u03a9-DP-52':'\u03a9-DP-52',
}
# Per-panel offsets: {short_name: {panel: (dx,dy)}}
# A: DP-19 (6.8,0.02), DP-52 (5.8,0.03) very close, DT-10 (6.8,-0.43) lower
# C: DP-19 (47,97), DP-52 (59,91), DT-10 (22,107)
# E: spread out
# C offsets: all arrows go rightward so lines never cross
# Top-to-bottom: DT-10 (22,107), DP-19 (47,97), DP-52 (59,91)
LEAD_OFFSETS={
    '\u03a9-DP-19':{'A':(12,10),'C':(18,-5),'E':(12,-10)},
    '\u03a9-DT-10':{'A':(12,-10),'C':(18,0),'E':(12,8)},
    '\u03a9-DP-52':{'A':(-45,-10),'C':(5,-14),'E':(12,-10)},
}

def annotate_lead(ax, x, y, label, offset=(12,8)):
    ax.annotate(label,(x,y),fontsize=6,ha='left',va='center',color='black',
                xytext=offset,textcoords='offset points',
                arrowprops=dict(arrowstyle='-',color='#555',lw=0.5),zorder=10)

STRAIN_MAP_ITALIC={
    'A. baumannii ATCC 19606 (-)':('$\\it{A. baumannii}$ ATCC 19606',False,'-'),
    'A. baumannii ATCC BAA-1605 (-) - CGTPACCIMRA':('$\\it{A. baumannii}$ BAA-1605 (CRAB)',True,'-'),
    'E. cloacae ATCC 13047 (-)':('$\\it{E. cloacae}$ ATCC 13047',False,'-'),
    'E. coli ATCC 11775 (-)':('$\\it{E. coli}$ ATCC 11775',False,'-'),
    'E. coli AIC221 (-)':('$\\it{E. coli}$ AIC221',False,'-'),
    'E. coli AIC222 - CRE (-)':('$\\it{E. coli}$ AIC222 (CRE)',True,'-'),
    'E. coli ATCC BAA-3170 (-) - CRE':('$\\it{E. coli}$ BAA-3170 (CRE)',True,'-'),
    'K. pneumoniae ATCC 13883 (-)':('$\\it{K. pneumoniae}$ ATCC 13883',False,'-'),
    'K. pneumoniae ATCC BAA-2342 (-) - EIRK':('$\\it{K. pneumoniae}$ BAA-2342 (ESBL)',True,'-'),
    'P. aeruginosa PAO1 (-)':('$\\it{P. aeruginosa}$ PAO1',False,'-'),
    'P. aeruginosa PA14 (-)':('$\\it{P. aeruginosa}$ PA14',False,'-'),
    'P. aeruginosa ATCC BAA-3197 (-) - FBCRP':('$\\it{P. aeruginosa}$ BAA-3197 (FQR)',True,'-'),
    'S. enterica ATCC 9150 (-)':('$\\it{S. enterica}$ ATCC 9150',False,'-'),
    'S. enterica Typhimurtium ATCC 700720':('$\\it{S. enterica}$ Typhimurium',False,'-'),
    'B. subtilis ATCC 23857 (+)':('$\\it{B. subtilis}$ ATCC 23857',False,'+'),
    'S. aureus ATCC 12600 (+)':('$\\it{S. aureus}$ ATCC 12600',False,'+'),
    'S. aureus ATCC BAA-1556 - MRSA (+)':('$\\it{S. aureus}$ BAA-1556 (MRSA)',True,'+'),
    'E. faecalis ATCC 700802 - VRE (+)':('$\\it{E. faecalis}$ 700802 (VRE)',True,'+'),
    'E. faecium ATCC 700221 - VRE (+)':('$\\it{E. faecium}$ 700221 (VRE)',True,'+'),
    'E. coli K-12 BW25113 (-)':('$\\it{E. coli}$ K-12 BW25113',False,'-'),
}
BENCH_COLORS={'OmegAMP':'#E8912D','AMP-Diffusion':'#2E5090','HydrAMP':'#00B4D8','CLaSS':'#2D6A4F','Joker':'#E63946'}
SS_COLORS={'Helix':'#5B8FB9','Beta':'#B6846C','Turn':'#7EB77F','Other':'#D1D1D1'}
print(f"De novo: {len(de_novo)} (P={len(dp)}, T={len(dt)})")

In [ ]:
fig=plt.figure(figsize=(6.5,8.0),dpi=300)
gs_top=gridspec.GridSpec(1,3,figure=fig,left=0.02,right=0.97,top=0.97,bottom=0.70,wspace=0.50)
gs_mid=gridspec.GridSpec(1,2,figure=fig,left=0.06,right=0.97,top=0.64,bottom=0.37,wspace=0.40,width_ratios=[1.2,1])
gs_bot=gridspec.GridSpec(1,2,figure=fig,left=0.02,right=0.97,top=0.31,bottom=0.04,wspace=0.40)
L=dict(fontsize=10,fontweight='bold',va='top',ha='left')

# A
ax=fig.add_subplot(gs_top[0]); ax.text(-0.14,1.06,'A',transform=ax.transAxes,**L)
sns.kdeplot(data=ref_amp_props,x='charge',y='hydrophobicity',levels=6,color='gray',linewidths=0.5,alpha=0.5,ax=ax)
for subset,marker,ms in [(dp,'o',16),(dt,'D',20)]:
    sc=ax.scatter(subset['net_charge'],subset['mean_hydrophobicity'],
                  c=subset['mic50'].clip(0,64).fillna(64),cmap=mic_cmap,norm=mic_norm,
                  marker=marker,s=ms,edgecolor='black',linewidth=0.3,alpha=0.85,zorder=3)
    for _,row in subset.iterrows():
        if row['short_name'] in LEADS:
            off=LEAD_OFFSETS[row['short_name']]['A']
            annotate_lead(ax,row['net_charge'],row['mean_hydrophobicity'],LEADS[row['short_name']],off)
cb=plt.colorbar(sc,ax=ax,shrink=1.0,pad=0.02,aspect=25)
cb.set_label('MIC$_{50}$ ($\\mu$M)',fontsize=6); cb.set_ticks([0,4,8,16,32,48,64]); cb.ax.tick_params(labelsize=6)
ax.set_xlabel('Net charge'); ax.set_ylabel('Hydrophobicity'); ax.set_xlim(-2,12); ax.set_ylim(-0.7,0.7)
ax.legend([Line2D([],[],marker='o',color='w',markerfacecolor='gray',markeredgecolor='k',ms=3,lw=0),
           Line2D([],[],marker='D',color='w',markerfacecolor='gray',markeredgecolor='k',ms=3,lw=0),
           Line2D([],[],color='gray',lw=0.7)],
          [f'OmegAMP-P (n={len(dp)})',f'OmegAMP-T (n={len(dt)})',f'Known AMPs (n={len(ref_amp_props):,})'],
          fontsize=6,loc='upper right',frameon=True,framealpha=0.9,edgecolor='#ddd')
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

# B
ax=fig.add_subplot(gs_top[1]); ax.text(-0.14,1.06,'B',transform=ax.transAxes,**L)
bs=bestsel.merge(ref[['short_name','category']],on='short_name')
bs_dn=bs[bs['category']=='de_novo'].sort_values('fH_SDS_H2O',ascending=False).reset_index(drop=True)
x=np.arange(len(bs_dn))
ax.bar(x,bs_dn['fH_SDS_H2O'],width=0.9,color=SS_COLORS['Helix'],label='Helix')
b=bs_dn['fH_SDS_H2O'].values.copy()
beta=(bs_dn['f_beta_anti_SDS_H2O']+bs_dn['f_beta_par_SDS_H2O']).values
ax.bar(x,beta,bottom=b,width=0.9,color=SS_COLORS['Beta'],label='Beta'); b+=beta
ax.bar(x,bs_dn['fturn_SDS_H2O'],bottom=b,width=0.9,color=SS_COLORS['Turn'],label='Turn'); b+=bs_dn['fturn_SDS_H2O'].values
ax.bar(x,bs_dn['fothers_SDS_H2O'],bottom=b,width=0.9,color=SS_COLORS['Other'],label='Other')
ax.set_ylabel('Secondary structure in SDS (%)'); ax.set_xlabel('Peptides (sorted by helicity)')
ax.set_xlim(-0.5,len(bs_dn)-0.5); ax.set_ylim(0,100); ax.set_xticks([])
ax.legend(fontsize=6,loc='center right',frameon=True,framealpha=0.9,edgecolor='#ddd',handlelength=1,handletextpad=0.3)

# C
ax=fig.add_subplot(gs_top[2]); ax.text(-0.14,1.06,'C',transform=ax.transAxes,**L)
dn_mech=de_novo.dropna(subset=['NPN','DiSC','mic50']).copy()
npn_med=dn_mech['NPN'].median(); disc_med=dn_mech['DiSC'].median()
sc=ax.scatter(dn_mech['NPN'],dn_mech['DiSC'],c=dn_mech['mic50'].clip(0,64).fillna(64),
              cmap=mic_cmap,norm=mic_norm,s=14,alpha=0.8,edgecolors='#666',linewidth=0.3,zorder=3)
ax.axvline(npn_med,color='#bbb',ls=':',lw=0.5); ax.axhline(disc_med,color='#bbb',ls=':',lw=0.5)
for (xf,yf,ha,va),cond in [
    ((0.03,0.97,'left','top'),(dn_mech['NPN']<npn_med)&(dn_mech['DiSC']>=disc_med)),
    ((0.97,0.97,'right','top'),(dn_mech['NPN']>=npn_med)&(dn_mech['DiSC']>=disc_med)),
    ((0.03,0.03,'left','bottom'),(dn_mech['NPN']<npn_med)&(dn_mech['DiSC']<disc_med)),
    ((0.97,0.03,'right','bottom'),(dn_mech['NPN']>=npn_med)&(dn_mech['DiSC']<disc_med)),
]:
    ax.text(xf,yf,str(cond.sum()),transform=ax.transAxes,fontsize=8,fontweight='bold',ha=ha,va=va,color='#aaa')
for _,row in dn_mech.iterrows():
    if row['short_name'] in LEADS:
        off=LEAD_OFFSETS[row['short_name']]['C']
        annotate_lead(ax,row['NPN'],row['DiSC'],LEADS[row['short_name']],off)
ax.set_xlabel('NPN MaxRel (%)'); ax.set_ylabel('DiSC$_3$(5) MaxRel (%)')
cb=plt.colorbar(sc,ax=ax,shrink=1.0,pad=0.02,aspect=25)
cb.set_label('MIC$_{50}$ ($\\mu$M)',fontsize=6); cb.set_ticks([0,4,8,16,32,48,64]); cb.ax.tick_params(labelsize=6)

# D
ax=fig.add_subplot(gs_mid[0]); ax.text(-0.11,1.06,'D',transform=ax.transAxes,**L)
mic_vals_dn=mic_dn[mic_strains].astype(float); n_total=len(mic_dn)
strain_data=[]
for s in mic_strains:
    vals=mic_vals_dn[s]; info=STRAIN_MAP_ITALIC.get(s,(s[:30],False,'-'))
    strain_data.append({'name':info[0],'potent':(vals<=2).sum()/n_total*100,'active':(vals<=4).sum()/n_total*100,'mdr':info[1],'gram':info[2]})
sdf=pd.DataFrame(strain_data).sort_values('potent',ascending=True).reset_index(drop=True)
y=np.arange(len(sdf))
ax.barh(y,sdf['active'],height=0.7,color='#FFD580',label='MIC $\\leq$ 4 $\\mu$M',zorder=2)
ax.barh(y,sdf['potent'],height=0.7,color='#E8912D',label='MIC $\\leq$ 2 $\\mu$M',zorder=3)
ax.set_yticks(y); ylabels=ax.set_yticklabels(sdf['name'],fontsize=6)
for i,(_,row) in enumerate(sdf.iterrows()):
    ylabels[i].set_color('#DC2626' if row['gram']=='+' else '#2563EB')
    if row['mdr']: ylabels[i].set_fontweight('bold')
ax.set_xlabel('Peptides with MIC $\\leq$ threshold (%)'); ax.set_xlim(0,105)
ax.legend(fontsize=6,loc='lower right',frameon=True,framealpha=0.9,edgecolor='#ddd')

# E
ax=fig.add_subplot(gs_mid[1]); ax.text(-0.11,1.06,'E',transform=ax.transAxes,**L)
dn_safe=de_novo.dropna(subset=['mic50','CC50']).copy()
dn_safe['log2_mic50']=np.log2(dn_safe['mic50'].clip(0.5))
dn_safe['log10_cc50']=np.log10(dn_safe['CC50'].clip(0.5))
sc=ax.scatter(dn_safe['log2_mic50'],dn_safe['log10_cc50'],c=dn_safe['n_strains_le2'],
              cmap='YlOrRd',s=14,alpha=0.8,edgecolors='#666',linewidth=0.3,vmin=0,vmax=18,zorder=3)
for _,row in dn_safe.iterrows():
    if row['short_name'] in LEADS:
        off=LEAD_OFFSETS[row['short_name']]['E']
        annotate_lead(ax,row['log2_mic50'],row['log10_cc50'],LEADS[row['short_name']],off)
lo_x,hi_x=int(np.floor(dn_safe['log2_mic50'].min())),int(np.ceil(dn_safe['log2_mic50'].max()))
ticks_x=list(range(lo_x,hi_x+1)); ax.set_xticks(ticks_x)
ax.set_xticklabels([str(int(2**t)) if t>=0 else f'{2**t:.1f}' for t in ticks_x])
lo_y,hi_y=int(np.floor(dn_safe['log10_cc50'].min())),int(np.ceil(dn_safe['log10_cc50'].max()))
ticks_y=list(range(lo_y,hi_y+1)); ax.set_yticks(ticks_y)
ax.set_yticklabels([f'$10^{t}$' if t!=0 else '1' for t in ticks_y])
ax.set_xlabel('MIC$_{50}$ ($\\mu$M, log$_2$)'); ax.set_ylabel('CC$_{50}$ ($\\mu$M, log$_{10}$)')
cb=plt.colorbar(sc,ax=ax,shrink=1.0,pad=0.02,aspect=25)
cb.set_label('Strains MIC $\\leq$ 2 $\\mu$M',fontsize=6); cb.ax.tick_params(labelsize=6)
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

# F
ax=fig.add_subplot(gs_bot[0]); ax.text(-0.11,1.06,'F',transform=ax.transAxes,**L)
try: ad=pd.read_csv(BENCH+'amp-diffusion.csv')
except FileNotFoundError: ad=pd.DataFrame()
if len(ad)>0:
    strain_map_ad={}
    for ad_strain in ad['strain'].unique():
        al=ad_strain.lower()
        for omega_col in mic_strains:
            ol=omega_col.lower()
            for key in ['19606','1605','11775','aic221','aic222','13883','pao1','pa14','12600','1556','700802','700221']:
                if key in al and key in ol: strain_map_ad[ad_strain]=omega_col; break
    shared=sorted(strain_map_ad.keys())
    for thresh,ls,alpha in [(4,'-',1.0),(2,'--',0.7)]:
        omega_c=[sum(1 for s in shared if float(r[strain_map_ad[s]])<=thresh) for _,r in mic_dn.iterrows()]
        ad_c=[]
        for pid in ad['peptide_id'].unique():
            pd_=ad[(ad['peptide_id']==pid)&(ad['mic_relation']=='=')]
            ad_c.append(sum(1 for s in shared if s in pd_['strain'].values and pd_[pd_['strain']==s]['mic'].values[0]<=thresh))
        x_r=range(1,len(shared)+1)
        ax.plot(x_r,[sum(c>=n for c in omega_c)/len(omega_c)*100 for n in x_r],
                f'o{ls}',color=BENCH_COLORS['OmegAMP'],ms=3,alpha=alpha,lw=1.2,label=f'OmegAMP MIC $\\leq$ {thresh}')
        ax.plot(x_r,[sum(c>=n for c in ad_c)/len(ad_c)*100 for n in x_r],
                f's{ls}',color=BENCH_COLORS['AMP-Diffusion'],ms=3,alpha=alpha,lw=1.0,label=f'AMP-Diff MIC $\\leq$ {thresh}')
    ax.set_xlabel(f'Minimum strains hit (of {len(shared)} shared)'); ax.set_ylabel('Peptides reaching threshold (%)')
    ax.legend(fontsize=6,loc='upper right',frameon=True,framealpha=0.9,edgecolor='#ddd')
    ax.set_xlim(0.5,len(shared)+0.5); ax.xaxis.set_major_locator(MaxNLocator(integer=True))

# G
ax=fig.add_subplot(gs_bot[1]); ax.text(-0.11,1.06,'G',transform=ax.transAxes,**L)
_AA_MASS={'A':71.0788,'R':156.1875,'N':114.1038,'D':115.0886,'C':103.1388,
          'Q':128.1307,'E':129.1155,'G':57.0519,'H':137.1411,'I':113.1594,
          'L':113.1594,'K':128.1741,'M':131.1926,'F':147.1766,'P':97.1167,
          'S':87.0782,'T':101.1051,'W':186.2132,'Y':163.1760,'V':99.1326}
def _seq_mw_kda(seq):
    if not isinstance(seq,str) or not any(c.isalpha() for c in seq): return 2.0
    return (sum(_AA_MASS.get(aa,111.1) for aa in seq.upper() if aa.isalpha())+18.0153)/1000
thresholds_uM=[1,2,4,8,16,32,64,128]
ecoli_cols=[c for c in mic_strains if 'E. coli' in c and 'K-12' not in c]
omega_best=mic_dn[ecoli_cols].astype(float).min(axis=1); n_omega=len(omega_best)
ax.plot(thresholds_uM,[(omega_best<=t).sum()/n_omega for t in thresholds_uM],
        'o-',color=BENCH_COLORS['OmegAMP'],ms=3,lw=1.5,label=f'OmegAMP (n={n_omega})',zorder=5)
for name,fn,marker in [('AMP-Diffusion','amp-diffusion.csv','s'),('HydrAMP','hydramp.csv','D'),
                        ('CLaSS','class.csv','v'),('Joker','joker.csv','^')]:
    try: comp=pd.read_csv(BENCH+fn)
    except FileNotFoundError: continue
    sp=comp[comp['strain'].str.contains('Escherichia coli|E. coli',case=False,na=False)]
    if len(sp)==0: continue
    if sp['mic_unit'].iloc[0]=='ug/mL':
        mw_map=comp.drop_duplicates('peptide_id').set_index('peptide_id')['sequence'].map(_seq_mw_kda)
        sp_eq=sp[sp['mic_relation']=='='].copy()
        sp_eq['mic_uM']=sp_eq.apply(lambda r:r['mic']/mw_map.get(r['peptide_id'],2.0),axis=1)
        best=sp_eq.groupby('peptide_id')['mic_uM'].min()
    else:
        best=sp[sp['mic_relation']=='='].groupby('peptide_id')['mic'].min()
    n_c=comp['peptide_id'].nunique()
    ax.plot(thresholds_uM,[(best<=t).sum()/n_c for t in thresholds_uM],
            f'{marker}--',color=BENCH_COLORS.get(name,'#999'),ms=3,lw=1.0,label=f'{name} (n={n_c})')
ax.set_xscale('log',base=2); ax.set_xticks(thresholds_uM); ax.set_xticklabels(thresholds_uM,fontsize=6)
ax.set_xlabel('MIC threshold ($\\mu$M)'); ax.set_ylabel('Fraction with $\\it{E. coli}$\nMIC $\\leq$ threshold')
ax.set_ylim(-0.02,1.05); ax.set_title('$\\it{E. coli}$',fontsize=7,pad=3)
leg=ax.legend(fontsize=6,loc='upper left',frameon=True,framealpha=0.8,edgecolor='#ccc',bbox_to_anchor=(0.0,0.95))
leg.get_frame().set_facecolor('white'); leg.set_zorder(10)

plt.savefig('/home/pszymczak/code/omegamp-dashboard/figures/figure4_denovo.pdf',bbox_inches='tight')
plt.savefig('/home/pszymczak/code/omegamp-dashboard/figures/figure4_denovo.svg',bbox_inches='tight')
plt.savefig('/home/pszymczak/code/omegamp-dashboard/figures/figure4_denovo.png',dpi=600,bbox_inches='tight')
plt.show()
print("Saved figure4_denovo")